# RAG (Retrieval-Augmented Generation) & MCP — 핵심 개념 및 정답 해설

원본 실습(`rag_raw_source` 내 6개 노트북)과 `평가체크포인트.xlsx`를 기반으로 구성된 19개 핵심 문항의 정답 및 해설입니다.

## 핵심 개념 1. (a-1) HTML 검색 결과 파싱 및 문장 단위 청크 추출 (parse_htmls & extract_chunks)

### 핵심 원리 및 코드 설명
- **HTML 정제 및 문장 단위 청킹(Chunking)**: 웹 검색 결과(`page_result`)에는 불필요한 HTML 태그가 포함되어 있어 그대로 LLM에 전달하면 컨텍스트 길이를 낭비하고 노이즈를 유발합니다.
- `BeautifulSoup.get_text(" ", strip=True)`로 본문 텍스트만 추출한 뒤, 글자 수로 단순 절단하면 문장이 중간에 끊겨 의미가 훼손되므로 `blingfire.text_to_sentences_and_offsets`를 사용해 문장 경계(`(start, end)`) 단위로 분할하고 `[:MAX_CONTEXT_SENTENCE_LENGTH]`로 상한 길이를 제한합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`soup.get_text(" ", strip=True)`)**:
   - **출제 포인트**: HTML DOM 트리에서 모든 태그를 제거하고 텍스트 노드들을 공백(`" "`)으로 연결하며 양끝 공백을 제거(`strip=True`)합니다.
2. **빈칸 2 (`text_to_sentences_and_offsets(document)`)**:
   - **출제 포인트**: 문서 문자열을 문장 단위로 분리하여 `(문장 텍스트, [(start, end), ...])` 튜플을 반환합니다.
3. **빈칸 3 (`document[start:end][:MAX_CONTEXT_SENTENCE_LENGTH]`)**:
   - **출제 포인트**: 오프셋 구간으로 한 문장을 추출한 뒤 최대 허용 글자 수(`MAX_CONTEXT_SENTENCE_LENGTH`)로 절단합니다.

In [ ]:
# [코드]
MAX_CONTEXT_SENTENCE_LENGTH = 1000

def parse_htmls(search_results):
    all_documents = []

    # 각 검색 결과의 HTML 원문에서 텍스트만 추출
    for html_text in search_results:
        # lxml 파서를 사용하여 HTML 파싱
        soup = BeautifulSoup(html_text["page_result"], features="lxml")
        # 공백(" ")을 구분자로 태그 내부 텍스트를 추출하고 앞뒤 공백 제거
        text = ### soup.get_text(" ", strip=True) ### 작성 필요 ###
        all_documents.append(text)

    return all_documents

def extract_chunks(all_documents):
    all_chunks = []

    for document in all_documents:
        if not document:
            all_chunks.append("")
        else:
            # blingfire를 이용해 문장별 (시작 인덱스, 끝 인덱스) 오프셋 추출
            _, offsets = ### text_to_sentences_and_offsets(document) ### 작성 필요 ###
            chunks = []

            for start, end in offsets:
                # 문장 구간을 슬라이싱하고 최대 길이(MAX_CONTEXT_SENTENCE_LENGTH)로 제한
                chunk = ### document[start:end][:MAX_CONTEXT_SENTENCE_LENGTH] ### 작성 필요 ###
                all_chunks.append(chunk)

    return all_chunks

## 핵심 개념 2. (b-1) LlamaIndex 문서 로드, 벡터 인덱스 및 Query Engine 생성

### 핵심 원리 및 코드 설명
- **LlamaIndex 기본 RAG 파이프라인 3단계**:
  1. **Ingestion (`SimpleDirectoryReader`)**: 로컬 디렉터리 내 텍스트/PDF/문서 파일들을 읽어 `Document` 객체 리스트로 변환합니다.
  2. **Indexing (`VectorStoreIndex.from_documents`)**: 문서를 노드(Node)로 분할하고 임베딩 벡터로 변환하여 인메모리 벡터 스토어에 인덱싱합니다.
  3. **Querying (`index.as_query_engine`)**: 검색기(Retriever)와 응답 합성기(Response Synthesizer)가 결합된 질의 엔진 인터페이스를 생성합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`SimpleDirectoryReader("data").load_data()`)**:
   - **출제 포인트**: `"data"` 폴더 내의 문서 파일들을 읽어 `Document` 리스트로 반환합니다.
2. **빈칸 2 (`VectorStoreIndex.from_documents(documents)`)**:
   - **출제 포인트**: `Document` 리스트로부터 노드 분할 및 임베딩을 수행하여 벡터 인덱스를 생성합니다.
3. **빈칸 3 (`index.as_query_engine(llm=llm)`)**:
   - **출제 포인트**: 생성된 벡터 인덱스에 응답 생성용 `llm`을 연결하여 질의 엔진 객체를 생성합니다.

In [ ]:
# [코드]
from llama_index.llms.openai import OpenAI as LlamaOpenAI

# "data" 디렉터리 내의 파일들을 읽어 Document 객체 리스트로 로드
documents = ### SimpleDirectoryReader("data").load_data() ### 작성 필요 ###

# 로드된 Document 리스트로부터 VectorStoreIndex 생성
index = ### VectorStoreIndex.from_documents(documents) ### 작성 필요 ###

# 응답 합성에 사용할 OpenAI LLM 설정 후 인덱스로부터 Query Engine 생성
llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0)
query_engine = ### index.as_query_engine(llm=llm) ### 작성 필요 ###

response = query_engine.query("What is the first programs the author tried writing?")

## 핵심 개념 3. (b-2) SentenceSplitter 청크 분할 및 transformations 기반 인덱스 생성

### 핵심 원리 및 코드 설명
- **토큰 기반 노드 분할 (`SentenceSplitter`)**: LlamaIndex의 `SentenceSplitter`는 문장 경계를 가급적 유지하면서 토큰 수 기준으로 문서를 `chunk_size` 크기의 노드(Node)로 분할하며, 인접 노드 간 문맥 단절을 막기 위해 `chunk_overlap`만큼의 토큰을 중복 포함시킵니다.
- 인덱스 생성 시 기본 설정 대신 커스텀 분할기를 적용하려면 `VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter])` 형태로 전달합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`SentenceSplitter(chunk_size=1024, chunk_overlap=200)`)**:
   - **출제 포인트**: 청크 최대 토큰 크기(1024)와 인접 청크 간 중첩 토큰 수(200)를 지정하여 분할기를 초기화합니다.
2. **빈칸 2 (`parser.get_nodes_from_documents(documents)`)**:
   - **출제 포인트**: `Document` 리스트를 분할하여 `TextNode` 객체들의 리스트를 반환합니다.
3. **빈칸 3 (`VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter])`)**:
   - **출제 포인트**: `transformations` 파이프라인에 커스텀 `text_splitter`를 지정하여 인덱스를 구축합니다.

In [ ]:
# [코드]
from llama_index.core.node_parser import SentenceSplitter

# 청크 크기 1024, 중첩 크기 200으로 문장 분할기 초기화
parser = ### SentenceSplitter(chunk_size=1024, chunk_overlap=200) ### 작성 필요 ###

# Document 리스트를 분할하여 Node 리스트 추출
nodes = ### parser.get_nodes_from_documents(documents) ### 작성 필요 ###

# 더 작은 청크 크기(200)와 중첩(50)을 갖는 분할기를 정의하고 transformations에 적용하여 인덱스 생성
text_splitter = SentenceSplitter(chunk_size=200, chunk_overlap=50)
index = ### VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter]) ### 작성 필요 ###

## 핵심 개념 4. (b-3) VectorStoreIndex 문서 동적 관리 (insert, update_ref_doc, delete_ref_doc)

### 핵심 원리 및 코드 설명
- **LlamaIndex 인덱스 동적 문서 관리 (CRUD)**:
  - **Insert (`index.insert(docu)`)**: 전체 인덱스를 재구축하지 않고 새 `Document`만 임베딩하여 기존 인덱스에 추가합니다.
  - **Update (`index.update_ref_doc(docu, ...)`)**: 동일한 `id_`(`doc_id`)를 가진 기존 문서를 찾아 변경된 텍스트로 재인덱싱합니다.
  - **Delete (`index.delete_ref_doc(id, delete_from_docstore=True)`)**: 지정한 `doc_id`에 해당하는 노드들을 인덱스 및 문서 저장소(`docstore`)에서 삭제합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`Document(text=data_text, id_="new_doc_id")`)**:
   - **출제 포인트**: 추후 업데이트 및 삭제 추적이 가능하도록 고유 `id_`를 부여하여 `Document` 객체를 생성합니다.
2. **빈칸 2 (`index.insert(docu)`)**:
   - **출제 포인트**: 생성된 `Document`를 기존 벡터 인덱스에 동적으로 삽입합니다.
3. **빈칸 3 (`index.delete_ref_doc(id, delete_from_docstore=True)`)**:
   - **출제 포인트**: 참조 문서 ID(`id`)를 기준으로 인덱스 및 내부 `docstore`에서 문서를 완전히 삭제합니다.

In [ ]:
# [코드]
# 고유 ID("new_doc_id")와 텍스트를 지정하여 새로운 Document 객체 생성
docu = ### Document(text=data_text, id_="new_doc_id") ### 작성 필요 ###

# 기존 인덱스에 새 문서 동적 삽입
### index.insert(docu) ### 작성 필요 ###

# 문서 내용을 수정한 뒤 동일 doc_id 기준으로 인덱스 업데이트
docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 2,000–6,000 °C and at pressures of 8–9 GPa.")
output = index.update_ref_doc(
    docu,
    update_kwargs={"delete_kwargs": {"delete_from_docstore": True}},
)

# 문서의 doc_id를 조회하여 인덱스 및 docstore에서 영구 삭제
id = docu.doc_id
### index.delete_ref_doc(id, delete_from_docstore=True) ### 작성 필요 ###

## 핵심 개념 5. (b-4) CustomQueryEngine 상속 기반 StandardQueryEngine 구현

### 핵심 원리 및 코드 설명
- **Query Engine의 내부 동작 분해 (`Retriever` + `ResponseSynthesizer`)**:
  - `CustomQueryEngine`을 상속받아 `custom_query(self, query_str: str)`를 오버라이딩하면 검색과 합성 단계를 세밀하게 제어할 수 있습니다.
  - 1단계로 `self.retriever.retrieve(query_str)`가 질의와 유사한 `NodeWithScore` 리스트를 반환하고, 2단계로 `self.response_synthesizer.synthesize(query_str, nodes)`가 검색된 노드들과 질의를 결합해 최종 응답을 생성합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`self.retriever.retrieve(query_str)`)**:
   - **출제 포인트**: 사용자 질의 문자열로 벡터 인덱스에서 관련 노드들을 검색합니다.
2. **빈칸 2 (`self.response_synthesizer.synthesize(query_str, nodes)`)**:
   - **출제 포인트**: 질의 문자열과 검색된 노드 리스트를 응답 합성기에 전달하여 최종 답변 객체를 생성합니다.
3. **빈칸 3 (`get_response_synthesizer(response_mode="compact", llm=llm)`)**:
   - **출제 포인트**: 검색된 청크들을 프롬프트 컨텍스트 한도 내에서 최대한 압축 결합(`response_mode="compact"`)하는 응답 합성기를 생성합니다.

In [ ]:
# [코드]
class StandardQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        # 1단계: Retriever를 사용하여 질의와 관련된 노드 리스트 검색
        nodes = ### self.retriever.retrieve(query_str) ### 작성 필요 ###
        # 2단계: Response Synthesizer에 질의와 검색된 노드를 전달하여 응답 합성
        response_obj = ### self.response_synthesizer.synthesize(query_str, nodes) ### 작성 필요 ###
        return response_obj

retriever = index.as_retriever()
# compact 모드로 동작하는 응답 합성기 생성
synthesizer = ### get_response_synthesizer(response_mode="compact", llm=llm) ### 작성 필요 ###
query_engine = StandardQueryEngine(
    retriever=retriever, response_synthesizer=synthesizer
)

## 핵심 개념 6. (b-5) PromptTemplate 포맷팅 및 OurCustomQueryEngine 구현

### 핵심 원리 및 코드 설명
- **커스텀 프롬프트 기반 Query Engine**: 기본 `ResponseSynthesizer` 대신 요약 길이 제한(예: 20단어 이내 요약)이나 엄격한 답변 규칙을 직접 적용하고 싶을 때 `PromptTemplate`을 정의합니다.
- `retriever.retrieve(query_str)`가 반환한 각 원소 `n`은 `NodeWithScore` 객체이므로 `n.node.get_content()`를 호출해 실제 텍스트를 꺼내고, `qa_prompt.format(context_str=..., query_str=...)`로 자리표시자를 치환한 뒤 `llm.complete()`를 호출합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`"\n\n".join([n.node.get_content() for n in nodes])`)**:
   - **출제 포인트**: 검색된 각 `NodeWithScore`에서 `.node.get_content()`로 텍스트를 추출한 뒤 빈 줄(`"\n\n"`)로 연결합니다.
2. **빈칸 2 (`self.qa_prompt.format(context_str=context_str, query_str=query_str)`)**:
   - **출제 포인트**: `PromptTemplate` 내의 `{context_str}`와 `{query_str}` 변수에 검색된 문맥과 사용자 질의를 바인딩합니다.

In [ ]:
# [코드]
simple_qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

class OurCustomQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OpenAI
    qa_prompt: PromptTemplate = simple_qa_prompt

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        # 검색된 각 노드의 텍스트(get_content)를 추출하여 줄바꿈("\n\n")으로 결합
        context_str = ### "\n\n".join([n.node.get_content() for n in nodes]) ### 작성 필요 ###
        # 프롬프트 템플릿에 문맥(context_str)과 질의(query_str)를 포맷팅하여 LLM 완성 호출
        response = self.llm.complete(
            ### self.qa_prompt.format(context_str=context_str, query_str=query_str) ### 작성 필요 ###
        )

        return str(response)

## 핵심 개념 7. (c-1) 프로토타입 RAG 클래스 구현 (RAG_from_scratch)

### 핵심 원리 및 코드 설명
- **RAG 파이프라인의 핵심 흐름 (Retrieve-then-Generate)**:
  - `org_query`는 외부 검색 문맥(`""`) 없이 LLM의 파라미터 지식만으로 답변하는 베이스라인(Non-RAG) 동작을 수행합니다.
  - 반면 `query` 메서드는 `self.retrieve(query)`로 외부 지식베이스에서 관련 정보를 먼저 가져온 뒤(`context_str`), 이를 `self.generate_response(query, context_str)`의 프롬프트 문맥에 주입하여 환각을 줄이고 최신/전문 사실에 기반한 답변을 생성합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`query_engine.query(query)`)**:
   - **출제 포인트**: 벡터 스토어 기반 질의 엔진에 쿼리를 전달하여 관련 문맥 정보를 검색합니다.
2. **빈칸 2 (`self.retrieve(query)`)**:
   - **출제 포인트**: RAG 질의 시 1단계로 검색 메서드를 호출해 `context_str`를 확보합니다.
3. **빈칸 3 (`self.generate_response(query, context_str)`)**:
   - **출제 포인트**: 원본 질문과 1단계에서 검색된 `context_str`를 함께 생성기에 전달하여 최종 답변을 얻습니다.

In [ ]:
# [코드]
class RAG_from_scratch:
    def retrieve(self, query: str) -> list:
        """
        Retrieve relevant text from vector store.
        """
        # 질의 엔진을 통해 질문과 관련된 문맥 정보 검색
        results = ### query_engine.query(query) ### 작성 필요 ###
        return results

    def generate_response(self, query: str, context_str: list) -> str:
        """
        Generate answer from context.
        """
        completion = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=
        [
            {"role": "user",
            "content":
            f"We have provided context information below. \n"
            f"---------------------\n"
            f"{context_str}"
            f"\n---------------------\n"
            f"Given this information, please answer the question: {query}"
            }
        ]
        ).choices[0].message.content
        return completion

    def org_query(self, query:str) -> str:
        completion = self.generate_response(query, "")
        return completion

    def query(self, query: str) -> str:
        # 1단계: 질의와 관련된 외부 문맥 검색
        context_str = ### self.retrieve(query) ### 작성 필요 ###
        # 2단계: 검색된 문맥과 질의를 함께 전달하여 최종 응답 생성
        completion = ### self.generate_response(query, context_str) ### 작성 필요 ###
        return completion

## 핵심 개념 8. (c-2) 청크 크기별 인덱스 구성 및 similarity_top_k Retriever 설정

### 핵심 원리 및 코드 설명
- **Chunk Size와 `similarity_top_k`의 트레이드오프**:
  - `chunk_size`가 너무 작으면(예: 200) 질문에 답하는 데 필요한 전후 문맥이 잘려 나가고, 너무 크면(예: 1024) 불필요한 잡음이 섞여 임베딩 유사도가 희석되거나 LLM 비용이 증가합니다.
  - `index.as_retriever(similarity_top_k=k)`는 코사인 유사도가 가장 높은 상위 `k`개의 노드만 검색하도록 제어하는 핵심 파라미터입니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter_short])`)**:
   - **출제 포인트**: 단문 분할기(`text_splitter_short`)를 변환 파이프라인에 적용해 벡터 인덱스를 생성합니다.
2. **빈칸 2 (`index_short.as_retriever(similarity_top_k=1)`)**:
   - **출제 포인트**: 유사도 상위 1개 노드만 반환하는 Retriever 객체를 생성합니다.

In [ ]:
# [코드]
# 단문 청크 분할기(200/50) 적용 인덱스 생성
text_splitter_short = SentenceSplitter(chunk_size=200, chunk_overlap=50)
index_short = ### VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter_short]) ### 작성 필요 ###

# 장문 청크 분할기(1024/200) 적용 인덱스 생성
text_splitter_long = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
index_long = VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter_long])

# 각 인덱스에서 가장 유사한 상위 1개(similarity_top_k=1) 노드만 검색하도록 Retriever 생성
retriever_short = ### index_short.as_retriever(similarity_top_k=1) ### 작성 필요 ###
retriever_long = index_long.as_retriever(similarity_top_k=1)

## 핵심 개념 9. (d-1) OpenAI 임베딩 및 코사인 유사도 기반 BaseRetriever 구현

### 핵심 원리 및 코드 설명
- **Dense Retrieval의 수학적 원리 (Cosine Similarity & Argsort)**:
  - $N$개 청크의 임베딩 행렬 $E \in \mathbb{R}^{N \times d}$(`all_embeddings`)와 질의 임베딩 $q \in \mathbb{R}^d$(`query_embedding`) 사이의 코사인 유사도는 다음과 같이 벡터화하여 계산합니다:
    $$\text{sim}(E, q) = \frac{E \cdot q}{\|E\|_2 \cdot \|q\|_2}$$
  - `np.argsort()`는 기본적으로 오름차순 인덱스를 반환하므로, 부호를 반전시킨 `-cosine_scores`에 `.argsort()[:topk]`를 적용하면 유사도가 가장 큰 상위 `topk`개의 인덱스를 정확히 얻을 수 있습니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`[np.array(item.embedding) for item in response.data]`)**:
   - **출제 포인트**: OpenAI 임베딩 응답 객체의 `response.data` 리스트에서 각 항목의 `.embedding` 벡터를 추출합니다.
2. **빈칸 2 (`np.dot(all_embeddings, query_embedding) / (np.linalg.norm(all_embeddings, axis=1) * np.linalg.norm(query_embedding))`)**:
   - **출제 포인트**: 행렬-벡터 내적을 각 행 벡터의 L2 노름(`axis=1`)과 질의 벡터 L2 노름의 곱으로 나누어 모든 청크의 코사인 유사도를 구합니다.
3. **빈칸 3 (`(-cosine_scores).argsort()[:topk]`)**:
   - **출제 포인트**: 음수 부호(`-`)를 이용해 내림차순 정렬 인덱스를 구하고 상위 `topk`개를 슬라이싱합니다.

In [ ]:
# [코드]
class BaseRetriever:
    def __init__(self,):
        self.client = openai.OpenAI(api_key = os.environ["OPENAI_API_KEY"])

    def embed_text(self, texts):
        """Generate embeddings using OpenAI's embedding model."""
        if isinstance(texts, str):
            texts = [texts]

        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )

        # 응답 객체(response.data)의 각 item에서 embedding 벡터를 NumPy 배열로 추출
        embeddings = ### [np.array(item.embedding) for item in response.data] ### 작성 필요 ###
        return np.array(embeddings)

    def retrieve(self, query, search_results, topk):
        all_documents = parse_htmls(search_results)
        all_chunks = extract_chunks(all_documents)

        # 전체 청크와 단일 질의에 대한 임베딩 생성
        all_embeddings = self.embed_text(all_chunks)
        query_embedding = self.embed_text(query)[0]

        # 내적(np.dot)을 각 벡터의 L2 노름(np.linalg.norm) 곱으로 나누어 코사인 유사도 계산
        cosine_scores = ### np.dot(all_embeddings, query_embedding) / (np.linalg.norm(all_embeddings, axis=1) * np.linalg.norm(query_embedding)) ### 작성 필요 ###
        # 유사도 내림차순(-cosine_scores) 정렬 후 상위 topk개 인덱스 추출
        top_k_indices = ### (-cosine_scores).argsort()[:topk] ### 작성 필요 ###
        top_k_chunks = np.array(all_chunks)[top_k_indices]

        return top_k_chunks

## 핵심 개념 10. (d-2) LlamaIndex 기반 검색기 (LlamaIndexRetriever) 구현

### 핵심 원리 및 코드 설명
- **LlamaIndex 기반 온디맨드(On-the-fly) 검색기**: 질의마다 주어진 웹 검색 결과(`search_results`)를 정제해 `Document` 객체로 래핑하고, `SentenceSplitter(chunk_size=512, chunk_overlap=0)`를 적용해 즉시 벡터 인덱스를 구축한 뒤 `similarity_top_k=topk`개의 가장 관련성 높은 청크 텍스트를 추출합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`VectorStoreIndex.from_documents(documents = documents, transformations=[self.parser])`)**:
   - **출제 포인트**: 생성된 `Document` 리스트에 `self.parser` 청크 분할기를 적용하여 벡터 인덱스를 구축합니다.
2. **빈칸 2 (`base_index.as_retriever(similarity_top_k=topk)`)**:
   - **출제 포인트**: 상위 `topk`개의 유사 노드를 반환하는 Retriever 인스턴스를 생성합니다.
3. **빈칸 3 (`[retrieved_node.node.get_content().strip() for retrieved_node in retrieved_nodes]`)**:
   - **출제 포인트**: 검색된 `NodeWithScore` 리스트에서 각 노드의 본문 문자열을 꺼내고 앞뒤 공백을 제거합니다.

In [ ]:
# [코드]
class LlamaIndexRetriever:
  def __init__(self):
      self.parser = SentenceSplitter(chunk_size=512, chunk_overlap=0)

  def retrieve(self, query, search_results, topk):
      documents = []

      for document in parse_htmls(search_results):
        if not document:
            documents.append(Document(text=""))
        else:
            documents.append(Document(text=document))

      # self.parser를 transformations로 전달하여 Document 리스트로부터 벡터 인덱스 생성
      base_index = ### VectorStoreIndex.from_documents(documents = documents, transformations=[self.parser]) ### 작성 필요 ###

      # 상위 topk개 노드를 검색하는 Retriever 생성 및 질의 실행
      base_retriever = ### base_index.as_retriever(similarity_top_k=topk) ### 작성 필요 ###
      retrieved_nodes = base_retriever.retrieve(query)

      # 검색된 각 노드에서 텍스트 본문을 추출하고 양끝 공백 제거
      retrieved_results = ### [retrieved_node.node.get_content().strip() for retrieved_node in retrieved_nodes] ### 작성 필요 ###

      return retrieved_results

## 핵심 개념 11. (d-3) 검색 청크 길이 제한 및 프롬프트 조합 기반 Reader 구현

### 핵심 원리 및 코드 설명
- **RAG Reader의 프롬프트 엔지니어링 및 컨텍스트 길이 제어**:
  - 환각(Hallucination)을 방지하기 위해 시스템 프롬프트에 *'참조 문서에 답이 없으면 반드시 I don\'t know라고 답하라'*는 지침(Abstention)을 명시합니다.
  - 검색된 청크들을 합친 문자열이 LLM 입력 토큰 한도를 초과하지 않도록 `references[:MAX_CONTEXT_REFERENCES_LENGTH]`로 절단한 뒤 `system` / `user` 역할 메시지 구조로 전달합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`self.prompt_generator(query, top_k_chunks)`)**:
   - **출제 포인트**: 질문과 검색된 청크 리스트를 결합하여 OpenAI 메시지 포맷(`llm_input`)을 생성합니다.
2. **빈칸 2 (`references[:MAX_CONTEXT_REFERENCES_LENGTH]`)**:
   - **출제 포인트**: 참조 문맥 문자열의 최대 글자 수를 `MAX_CONTEXT_REFERENCES_LENGTH`(4000자)로 제한합니다.
3. **빈칸 3 (`[{"role": "system", "content": self.system_prompt}, {"role": "user", "content": user_message}]`)**:
   - **출제 포인트**: Chat Completions API 규격에 맞춰 시스템 지침과 사용자 프롬프트 딕셔너리 리스트를 구성합니다.

In [ ]:
# [코드]
MAX_CONTEXT_REFERENCES_LENGTH = 4000

class Reader:
  def __init__(self):
    self.system_prompt = """
    You are provided with a question and various references.
    Your task is to answer the question succinctly, using the fewest words possible.
    If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
    There is no need to explain the reasoning behind your answers.
    """

  def generate_response(self, query: str, top_k_chunks: list) -> str:
      # 질문과 검색된 청크를 결합하여 LLM 입력 메시지 리스트 생성
      llm_input = ### self.prompt_generator(query, top_k_chunks) ### 작성 필요 ###
      completion = oai_client.chat.completions.create(
      model="gpt-3.5-turbo",
      temperature=0,
      messages=
      llm_input
      ).choices[0].message.content
      return completion

  def prompt_generator(self, query, top_k_chunks):
      user_message = ""
      references = ""

      if len(top_k_chunks) > 0:
          references += "# References \n"
          for chunk_id, chunk in enumerate(top_k_chunks):
              references += f"- {chunk.strip()}\n"

      # 모델 입력 크기 제한을 위해 참조 텍스트 길이를 MAX_CONTEXT_REFERENCES_LENGTH로 슬라이싱
      references = ### references[:MAX_CONTEXT_REFERENCES_LENGTH] ### 작성 필요 ###

      user_message += f"{references}\n------\n\n"
      user_message += f"Using only the references listed above, answer the following question: \n"
      user_message += f"Question: {query}\n"

      # system 역할과 user 역할 메시지 리스트 구성
      llm_input = ### [{"role": "system", "content": self.system_prompt}, {"role": "user", "content": user_message}] ### 작성 필요 ###

      return llm_input

## 핵심 개념 12. (d-4) Retriever와 Reader를 결합한 모듈형 RAG 클래스 구현

### 핵심 원리 및 코드 설명
- **모듈형 RAG 아키텍처**: 검색기(`Retriever`)와 독해기(`Reader`)를 독립된 컴포넌트로 분리하면 검색 알고리즘(Dense, BM25, Hybrid, KG)이나 리더 모델(GPT-3.5, GPT-4o)을 서로 독립적으로 교체·평가할 수 있습니다.
- `inference`가 `(answer, retrieved_results)`를 함께 반환하도록 설계하면 추후 답변 오류 발생 시 검색 실패(Retrieval Failure)인지 독해 실패(Reader Failure)인지 원인을 분석할 수 있습니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`self.retriever.retrieve(query, search_results, topk)`)**:
   - **출제 포인트**: 주어진 검색 결과 HTML로부터 질의와 가장 관련 높은 상위 `topk`개 청크 리스트를 추출합니다.
2. **빈칸 2 (`self.reader.generate_response(query, retrieved_results)`)**:
   - **출제 포인트**: 추출된 청크 리스트를 참조 문맥으로 삼아 질문에 대한 최종 답변을 생성합니다.

In [ ]:
# [코드]
class RAG:
    def __init__(self):
        self.retriever = LlamaIndexRetriever()
        self.reader = Reader()

    def inference(self, query, search_results, topk):
        # 1. Retriever를 사용해 관련 상위 topk개 청크 검색
        retrieved_results = ### self.retriever.retrieve(query, search_results, topk) ### 작성 필요 ###

        # 2. 검색된 청크를 Reader에 전달하여 최종 답변 생성
        answer = ### self.reader.generate_response(query, retrieved_results) ### 작성 필요 ###

        return answer, retrieved_results

## 핵심 개념 13. (e-1) Mock KG API 연동을 위한 도메인 판별 및 JSON 쿼리 추출 (generate_query)

### 핵심 원리 및 코드 설명
- **지식 그래프(KG) 질의를 위한 구조화 엔티티 추출**: 정형 API(예: 주가, 배당금, PER, EPS 조회)를 호출하려면 비정형 자연어 질문에서 `domain`, `market_identifier`, `metric`, `datetime`을 정형 JSON으로 먼저 추출해야 합니다.
- LLM 출력에 JSON 외 여분의 텍스트가 섞일 경우를 대비해 `json.loads(completion)` 실패 시 `extract_json_objects(completion)` 폴백(fallback)을 수행하고, `domain == "finance"` 여부로 금융 KG 호출 여부를 라우팅합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`json.loads(completion)`)**:
   - **출제 포인트**: LLM이 반환한 JSON 형식 문자열을 파이썬 딕셔너리 객체로 역직렬화합니다.
2. **빈칸 2 (`domain == "finance"`)**:
   - **출제 포인트**: 추출된 도메인이 금융(`"finance"`)인지 여부를 판별하여 불리언 플래그 `is_finance`를 설정합니다.

In [ ]:
# [코드]
def generate_query(query):
    llm_input = prompt_generator(query)
    completion = oai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=
    llm_input
    ).choices[0].message.content

    try:
        # LLM 출력 문자열을 JSON 딕셔너리로 파싱
        completion = ### json.loads(completion) ### 작성 필요 ###
    except:
        completion = extract_json_objects(completion)

    if "domain" in completion.keys():
        domain = completion["domain"]
        # 도메인이 "finance"인지 여부를 불리언 값으로 판별
        is_finance = ### domain == "finance" ### 작성 필요 ###
    else:
        is_finance = False

    return completion, is_finance

## 핵심 개념 14. (e-2) 웹 검색 결과와 지식 그래프(KG)를 결합한 하이브리드 RAG (RAGWithSRKG)

### 핵심 원리 및 코드 설명
- **하이브리드(Web + Knowledge Graph) RAG**: 영화·일반 상식 같은 서술형 질문은 웹 검색 문서(`retrieved_results`)가 유리하고, 특정 일자의 주가·PER·배당일 같은 정형 수치는 지식 그래프 API(`kg_results`)가 정확합니다.
- `RAGWithSRKG`는 도메인 판별 플래그(`is_finance`)에 따라 금융 질의에는 정형 KG 결과를, 비금융 질의에는 웹 문서 청크를 `Reader`에 전달하여 두 소스의 장점을 결합합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`self.kg_query_engine.query(query)`)**:
   - **출제 포인트**: 질문을 분석해 금융 KG API를 조회하고 `(kg_results, is_finance)` 튜플을 반환받습니다.
2. **빈칸 2 (`[kg_results]`)**:
   - **출제 포인트**: 금융 도메인 질문일 때 지식 그래프 검색 결과를 리스트 형태로 감싸 `combined_results`에 할당합니다.
3. **빈칸 3 (`self.reader.generate_response(query, combined_results)`)**:
   - **출제 포인트**: 선택된 참조 문맥(`combined_results`)을 Reader에 전달해 최종 응답을 생성합니다.

In [ ]:
# [코드]
class RAGWithSRKG:
    def __init__(self):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine()
        self.reader = Reader()

    def inference(self, query, search_results, topk):
        # 1. 웹 검색 결과에서 관련 청크 검색
        retrieved_results = self.retriever.retrieve(query, search_results, topk)

        # 2. 지식 그래프(KG) 질의 엔진을 통해 KG 결과 및 금융 도메인 여부 획득
        kg_results, is_finance = ### self.kg_query_engine.query(query) ### 작성 필요 ###

        # 금융 도메인이면 KG 결과를, 아니면 웹 검색 청크를 참조 문맥으로 선택
        if is_finance:
          combined_results = ### [kg_results] ### 작성 필요 ###
        else:
          combined_results = retrieved_results

        # 3. 선택된 참조 문맥을 바탕으로 최종 답변 생성
        answer = ### self.reader.generate_response(query, combined_results) ### 작성 필요 ###

        return answer, combined_results

## 핵심 개념 15. (f-1) CRAG 벤치마크 평가 지표 및 CRAG_score 계산

### 핵심 원리 및 코드 설명
- **CRAG 평가 메트릭과 환각 페널티(Hallucination Penalty)**:
  - 일반적인 QA 지표와 달리 CRAG 벤치마크는 근거 없는 오답(Hallucination/Incorrect)에 **-1점**의 감점을 부과하고, 모를 때 `'I don\'t know'`라고 기권(Missing)하면 **0점**을 부여합니다:
    $$\text{CRAG Score} = 1.0 \times N_{\text{exact}} + 0.5 \times N_{\text{acceptable}} + 0.0 \times N_{\text{missing}} - 1.0 \times N_{\text{hallucinate}}$$
  - 따라서 근거가 부족할 때 추측해서 틀리는 것(-1점)보다 기권하는 것(0점)이 전체 점수를 크게 높입니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`"i don't know" in prediction_lowercase`)**:
   - **출제 포인트**: 모델이 답변을 기권(Missing)했는지 판별하여 `n_miss` 카운트를 올립니다.
2. **빈칸 2 (`(len(finance_test_dataset_ids) - n_correct_exact - n_correct - n_miss)`)**:
   - **출제 포인트**: 전체 평가 샘플 중 정답(Exact/Acceptable)도 아니고 기권(Missing)도 아닌 샘플 수를 오답/환각(`n_hallucinate`) 개수로 산출합니다.
3. **빈칸 3 (`n_correct_exact + 0.5*n_correct - n_hallucinate`)**:
   - **출제 포인트**: CRAG 공식 가중치(+1, +0.5, -1)를 적용하여 총점을 계산합니다.

In [ ]:
# [코드]
n_miss, n_correct, n_correct_exact = 0, 0, 0

for data in tqdm(dataset):
  if data['interaction_id'] not in finance_test_dataset_ids:
    continue

  question = data['query']
  ground_truth_lowercase = str(data['answer']).strip().lower()
  web_search_results = data['search_results']

  prediction_lowercase = rag.inference(question, web_search_results, 5)['answer'].lower()

  # 정답과 정확히 일치하는 경우 (Exact Match)
  if prediction_lowercase == ground_truth_lowercase:
      n_correct_exact += 1
      continue
  # 모른다고 기권("i don't know")한 경우 (Missing)
  elif ### "i don't know" in prediction_lowercase ### 작성 필요 ###:
      n_miss += 1
      continue

  # LLM 심판(Judge)을 통한 허용 가능 정답(Acceptable) 여부 평가
  acceptable = CRAG_evaluation(question, ground_truth_lowercase, prediction_lowercase)

  if acceptable == 1:
    n_correct += 1

# 전체 평가 개수에서 exact, correct, miss를 제외한 나머지를 환각(오답) 개수로 계산
n_hallucinate = ### (len(finance_test_dataset_ids) - n_correct_exact - n_correct - n_miss) ### 작성 필요 ###

# CRAG 점수 계산 (+1.0 * exact + 0.5 * acceptable - 1.0 * hallucinate)
CRAG_score = ### n_correct_exact + 0.5*n_correct - n_hallucinate ### 작성 필요 ###

## 핵심 개념 16. (f-2) 자기 검증(Self-Verification) 단계가 추가된 L2 RAG 구현 (verify_answer & infer_L2)

### 핵심 원리 및 코드 설명
- **L2 Self-Verification & Abstention (자기 검증 및 기권)**:
  - 단일 패스 RAG(L1)는 검색된 문서에 근거가 부족해도 그럴듯한 오답을 지어내어 CRAG 점수에서 `-1` 감점을 받기 쉽습니다.
  - `verify_answer`는 질문, 참조 문서, 후보 답변을 Verifier LLM에 보여주고 참조 문서가 답변을 뒷받침하는지(`supported: true/false`)와 부족한 정보(`missing_information`)를 JSON으로 판정하게 합니다.
  - `supported`가 `False`일 때 `"I don't know"`로 대체하면 환각으로 인한 `-1`점을 `0`점(Missing)으로 방어할 수 있습니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`{"type": "json_object"}`)**:
   - **출제 포인트**: OpenAI Chat Completions의 출력 형식을 JSON 객체로 강제하는 `response_format` 설정입니다.
2. **빈칸 2 (`bool(parsed.get("supported", False)), str(parsed.get("missing_information", ""))`)**:
   - **출제 포인트**: 검증 결과 JSON에서 근거 뒷받침 여부(`supported`)와 누락된 정보(`missing_information`)를 추출해 반환합니다.
3. **빈칸 3 (`out["answer"] if supported else "I don't know"`)**:
   - **출제 포인트**: 검증을 통과한 경우에만 원래 답변을 채택하고, 그렇지 않으면 기권 응답을 반환합니다.

In [ ]:
# [코드]
def verify_answer(question, references, answer):
    user_message = (
        f"Question: {question}\n\n"
        f"# References\n{format_references(references)}\n\n"
        f"Candidate Answer: {answer}\n"
    )
    raw = oai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        # LLM이 반드시 JSON 객체를 반환하도록 response_format 지정
        response_format=### {"type": "json_object"} ### 작성 필요 ###,
        messages=[
            {"role": "system", "content": VERIFY_PROMPT},
            {"role": "user", "content": user_message},
        ],
    ).choices[0].message.content

    parsed = json_from_text(raw, {"supported": False, "missing_information": ""})
    # supported 불리언 여부와 missing_information 문자열을 튜플로 반환
    return ### bool(parsed.get("supported", False)), str(parsed.get("missing_information", "")) ### 작성 필요 ###


def infer_L2(item):
    out = rag.inference(item["query"], item["search_results"], 5)
    supported, _ = verify_answer(item["query"], out["retrieved_results"], out["answer"])
    # 검증 통과 시 답변 반환, 실패 시 "I don't know"로 기권
    return ### out["answer"] if supported else "I don't know" ### 작성 필요 ###

## 핵심 개념 17. (f-3) Agentic RAG 루프 구현 (Plan -> Act -> Read -> Verify)

### 핵심 원리 및 코드 설명
- **L3 Agentic RAG 루프의 4대 핵심 요소**:
  1. **Routing (`plan` -> `act`)**: 질문 성격과 이전 실패 원인에 따라 매 턴마다 웹 검색(`web_search`)과 금융 지식 그래프(`kg_lookup`) 중 적합한 도구를 선택합니다.
  2. **Query Rewriting**: 원래 질문을 그대로 재검색하지 않고, Verifier가 지적한 부족한 정보(`critique = missing`)를 타겟으로 새로운 검색어(`tool_query`)를 작성합니다.
  3. **Dynamic Stopping (`if supported: break`)**: 첫 턴에 검증을 통과한 쉬운 질문은 즉시 종료하여 비용을 아끼고, 부족한 경우에만 최대 `max_steps`까지 추가 검색합니다.
  4. **Safe Abstention (`for ... else`)**: 모든 턴을 소진해도 검증을 통과하지 못하면 `"I don't know"`로 안전하게 기권합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`list(self.retriever.retrieve(tool_query, search_results, self.topk))`)**:
   - **출제 포인트**: 플래너가 `web_search`를 선택했을 때 재작성된 `tool_query`로 웹 문서를 검색합니다.
2. **빈칸 2 (`evidence.extend(self.act(action, tool_query, search_results))`)**:
   - **출제 포인트**: 선택된 액션을 실행해 얻은 새 근거들을 누적 근거 리스트(`evidence`)에 이어 붙입니다.
3. **빈칸 3 (`verify_answer(query, evidence, answer)`)**:
   - **출제 포인트**: 현재까지 모인 `evidence`가 후보 `answer`를 뒷받침하는지 검증하고 누락된 정보(`missing`)를 받아옵니다.

In [ ]:
# [코드]
class AgenticRAG:
    def __init__(self, server=None, max_steps=3, topk=5, verbose=False):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine(server=server)
        self.reader = Reader()
        self.max_steps = max_steps
        self.topk = topk
        self.verbose = verbose
        self._reader_takes_time = (
            len(inspect.signature(self.reader.generate_response).parameters) >= 3
        )

    # [참고 메서드 정의 - plan / read 세부 로직 비공개]
    # ### 공개 제한 ###
    def plan(self, question, query_time, evidence, critique, tried):
        user_message = (
            f"Question: {question}\n"
            f"Query Time: {query_time}\n\n"
            f"# Evidence gathered so far\n{format_references(evidence[-5:], per_chunk=300)}\n\n"
            f"# Queries already tried\n{tried or '(none)'}\n\n"
            f"# Why the previous attempt was insufficient\n{critique or '(first turn)'}\n"
        )
        raw = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": PLANNER_PROMPT},
                {"role": "user", "content": user_message},
            ],
        ).choices[0].message.content

        plan = json_from_text(raw, {})
        action = plan.get("action", "web_search")
        if action not in ("web_search", "kg_lookup", "answer"):
            action = "web_search"
        return action, str(plan.get("query") or question), str(plan.get("reason", ""))

    def act(self, action, tool_query, search_results):
        if action == "web_search":
            # 재작성된 tool_query를 사용해 웹 검색 결과에서 상위 topk개 청크 검색
            return ### list(self.retriever.retrieve(tool_query, search_results, self.topk)) ### 작성 필요 ###
        if action == "kg_lookup":
            kg_results = self.kg_query_engine.query(tool_query)
            return [kg_results] if kg_results else []
        return []

    def read(self, question, query_time, evidence):
        if self._reader_takes_time:
            return self.reader.generate_response(question, query_time, evidence)
        return self.reader.generate_response(question, evidence)

    def inference(self, query, search_results, query_time=None):
        evidence, tried, steps = [], [], []
        critique, answer = None, "I don't know"

        for step in range(self.max_steps):
            action, tool_query, reason = self.plan(query, query_time, evidence, critique, tried)

            if action == "answer" and not evidence:
                action = "web_search"

            if action != "answer":
                tried.append(f"{action}: {tool_query}")
                # 선택된 도구를 실행하여 얻은 결과를 누적 evidence 리스트에 추가
                ### evidence.extend(self.act(action, tool_query, search_results)) ### 작성 필요 ###

            answer = self.read(query, query_time, evidence)
            # 누적된 evidence가 answer를 뒷받침하는지 검증
            supported, missing = ### verify_answer(query, evidence, answer) ### 작성 필요 ###

            steps.append({"step": step, "action": action, "query": tool_query, "reason": reason,
                          "answer": answer, "supported": supported, "missing": missing})

            if supported:
                break
            critique = missing
        else:
            answer = "I don't know"

        return {"answer": answer, "evidence": evidence, "steps": steps}

## 핵심 개념 18. (f-4) MCP 클라이언트 연결 및 FunctionAgent 기반 KGQueryEngineWithMCP 구현

### 핵심 원리 및 코드 설명
- **MCP (Model Context Protocol) 기반 동적 Tool Calling**:
  - 수동 의사결정 트리(Decision Tree)를 하드코딩하는 대신, MCP 서버가 제공하는 도구 스키마를 `McpToolSpec(client=BasicMCPClient(uri)).to_tool_list_async()`로 동적으로 가져옵니다.
  - `FunctionAgent`에 이 도구 리스트(`tools`)와 LLM을 등록하고 `self.agent.run(question, ctx=self.agent_context)`를 실행하면, 에이전트가 질문 해결에 필요한 KG 도구(`get_company_name`, `get_ticker_by_name`, `get_pe_ratio` 등)를 스스로 선택·호출하며 스트리밍 이벤트를 생성합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`McpToolSpec(client=mcp_client)`)**:
   - **출제 포인트**: MCP 클라이언트를 LlamaIndex 에이전트가 사용할 수 있는 도구 명세 객체(`McpToolSpec`)로 래핑합니다.
2. **빈칸 2 (`await self.mcp_tool_spec.to_tool_list_async()`)**:
   - **출제 포인트**: MCP 서버에 등록된 도구 목록과 메타데이터를 비동기로 조회하여 리스트로 변환합니다.
3. **빈칸 3 (`self.agent.run(question, ctx=self.agent_context)`)**:
   - **출제 포인트**: 에이전트 워크플로우 컨텍스트(`ctx`)와 함께 질문을 전달하여 비동기 에이전트 실행 핸들러를 시작합니다.

In [ ]:
# [코드]
external_mcp_server = "http://10.2.0.165:8081/sse"

# MCP 서버 주소로 클라이언트를 생성하고 McpToolSpec으로 래핑
mcp_client = BasicMCPClient(external_mcp_server)
mcp_tool = ### McpToolSpec(client=mcp_client) ### 작성 필요 ###

class KGQueryEngineWithMCP:
    def __init__(self, mcp_tool_spec: McpToolSpec, model: str, llm = None):
        self.llm = llm or OpenAI(model=model, temperature=0)
        self.mcp_tool_spec = mcp_tool_spec
        self.agent = None
        self.agent_context = None

    async def init_agent(self):
        # MCP 서버로부터 사용 가능한 도구(Tool) 리스트를 비동기로 로드
        tools = ### await self.mcp_tool_spec.to_tool_list_async() ### 작성 필요 ###
        for tool in tools:
            if len(tool.metadata.description) > 1000:
                tool.metadata.description = tool.metadata.description[:1000] + "..."
        self.agent = FunctionAgent(
            name="Agent",
            description="An agent that can work with Our Knowledge Graph api.",
            tools=tools,
            llm=self.llm,
            system_prompt=SYSTEM_PROMPT,
        )
        self.agent_context = Context(self.agent)

    async def query(self, question: str, verbose: bool = False) -> str:
        if self.agent is None or self.agent_context is None:
            raise RuntimeError("Agent not initialized. Call `await init_agent()` first.")

        # 공유 컨텍스트(ctx)와 함께 에이전트 실행 핸들러 시작
        handler = ### self.agent.run(question, ctx=self.agent_context) ### 작성 필요 ###
        async for event in handler.stream_events():
            if verbose and type(event) == ToolCall:
                print(f"Calling tool {event.tool_name} with kwargs {event.tool_kwargs}")
            elif verbose and type(event) == ToolCallResult:
                print(f"Tool {event.tool_name} returned {event.tool_output}")

        response = await handler
        return str(response)

## 핵심 개념 19. (f-5) MCP 에이전트와 시점 인지형 Reader를 결합한 RAGwithMCP 구현

### 핵심 원리 및 코드 설명
- **MCP 기반 RAG 통합 시 주의사항 (비동기 호출 및 참조 리스트 타입 보장)**:
  - 상대적 시간 표현('지난 분기', '올해')이 포함된 금융 질의를 정확히 처리하려면 `Query time: {query_time}`을 에이전트와 Reader 양쪽에 모두 전달해야 합니다.
  - `KGQueryEngineWithMCP.query()`는 문자열(`str`)을 반환하는 반면, `Reader.prompt_generator()`는 `top_k_chunks`를 순회(`for chunk in top_k_chunks`)합니다. 만약 문자열을 리스트로 감싸지 않고 그대로 넘기면 글자 하나하나(`'- M'`, `'- i'`)로 쪼개지는 버그가 발생하므로 `isinstance(retrieved_results, list)` 체크를 통해 반드시 리스트(`[retrieved_results]`)로 변환해야 합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`await self.mcp_application.query(full_query, verbose=False)`)**:
   - **출제 포인트**: 질의 시점 지침이 포함된 `full_query`를 MCP 에이전트에 비동기로 전달하여 도구 호출 결과를 얻습니다.
2. **빈칸 2 (`retrieved_results if isinstance(retrieved_results, list) else [retrieved_results]`)**:
   - **출제 포인트**: 반환된 검색 결과가 문자열일 때 글자 단위로 순회되는 것을 막기 위해 단일 원소 리스트로 감쌉니다.
3. **빈칸 3 (`await self.retrieve(query, query_time, search_results, topk)`)**:
   - **출제 포인트**: 비동기 검색 메서드를 `await`로 호출하여 MCP 기반 지식 검색 결과를 획득합니다.

In [ ]:
# [코드]
class RAGwithMCP:
    def __init__(self, mcp_tool):
        self.mcp_application = KGQueryEngineWithMCP(mcp_tool, model='gpt-4o-mini')
        self.reader = Reader()

    async def retrieve(self, query: str, query_time: str, search_results: list, topk: int):
        await self.mcp_application.init_agent()

        full_query = f"""Query: {query} When inferring time information, rely solely on the query time. Do not infer the time scope from the query itself.
Query time: {query_time}"""

        # MCP 에이전트에 full_query를 비동기로 질의하여 검색 결과 획득
        mcp_result = ### await self.mcp_application.query(full_query, verbose=False) ### 작성 필요 ###
        return mcp_result

    def generate_response(self, query: str, query_time: str, retrieved_results):
        # Reader가 문자열을 글자 단위로 순회하지 않도록 리스트가 아니면 [retrieved_results]로 래핑
        references = (### retrieved_results if isinstance(retrieved_results, list) else [retrieved_results] ### 작성 필요 ###)
        answer = self.reader.generate_response(query, query_time, references)
        return answer

    async def inference(self, query: str, search_results: list, query_time: str, topk: int):
        # 1단계: 비동기 MCP 검색 수행
        retrieved_results = ### await self.retrieve(query, query_time, search_results, topk) ### 작성 필요 ###
        # 2단계: 검색 결과를 바탕으로 Reader 응답 생성
        answer = self.generate_response(query, query_time, retrieved_results)
        return {
            "retrieved_results": retrieved_results,
            "answer": answer
        }